#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

#`Setup And Update ComfyUI`



In [ ]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = ""  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:

    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}

# `Models Download`

## Download MODELS (manifest các model đang dùng — HF direct)

In [ ]:
#@title Download model đang dùng — Hugging Face direct (verified)
%cd {WORKSPACE}
# Install Aria2
!apt-get -y install -qq aria2

import os
from google.colab import userdata

CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')  # optional: chỉ cần cho link civitai.com

# Nhóm model đang dùng — đặt True/False cho nhóm cần tải (~76G nếu bật tất!)
WANT = {
  "qwen-image":    True,   # ~28.7G — Qwen-Image-2512 fp8 + qwen2.5-vl 7B encoder + VAE + Lightning LoRA
  "trellis":       True,   # ~6.8G  — TRELLIS.2 int8 + shape/texture VAE
  "pixal3d":       True,   # ~6.3G  — Pixal3D int8 + dino_v3 clip_vision
  "3d-aux":        True,   # ~9.7G  — qwen3vl 8B encoder + birefnet + moge2 (aux cho TRELLIS.2/Pixal3D)
  "acestep":       True,   # ~9.4G  — ACE-Step 1.5 turbo + qwen 0.6B/1.7B encoders + VAE
  "minimax-music": True,   # ~4.8G  — MiniMax Music 3 DiT fp16 + dav VAE
  "stable-audio":  True,   # ~9.7G  — Stable Audio 3 medium + t5gemma encoder
  "qwen3.5":       False,  # ~4.2G  — qwen3.5 2B bf16 text encoder
}

# (group, models/<subdir>, filename, url, ~GB)
MODELS = [
  ("qwen-image", "diffusion_models", "qwen_image_2512_fp8_e4m3fn.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors", 19.0),
  ("qwen-image", "text_encoders", "qwen_2.5_vl_7b_fp8_scaled.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors", 8.7),
  ("qwen-image", "vae", "qwen_image_vae.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors", 0.2),
  ("qwen-image", "loras", "Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors",
   "https://huggingface.co/lightx2v/Qwen-Image-2512-Lightning/resolve/main/Qwen-Image-2512-Lightning-4steps-V1.0-bf16.safetensors", 0.8),

  ("trellis", "diffusion_models", "trellis_2_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/diffusion_models/trellis_2_int8_convrot.safetensors", 4.9),
  ("trellis", "vae", "trellis_2_shape_vae_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/vae/trellis_2_shape_vae_bf16.safetensors", 1.0),
  ("trellis", "vae", "trellis_2_texture_vae_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/TRELLIS.2/resolve/main/vae/trellis_2_texture_vae_bf16.safetensors", 0.9),

  ("pixal3d", "diffusion_models", "pixal3d_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/Pixal3D/resolve/main/diffusion_models/pixal3d_int8_convrot.safetensors", 5.2),
  ("pixal3d", "clip_vision", "dino_v3_L_naf_fp32.safetensors",
   "https://huggingface.co/Comfy-Org/Pixal3D/resolve/main/clip_vision/dino_v3_L_naf_fp32.safetensors", 1.1),

  ("3d-aux", "text_encoders", "qwen3vl_8b_int8_convrot.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen3-VL/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors", 8.7),
  ("3d-aux", "background_removal", "birefnet.safetensors",
   "https://huggingface.co/Comfy-Org/BiRefNet/resolve/main/background_removal/birefnet.safetensors", 0.4),
  ("3d-aux", "geometry_estimation", "moge_2_vitl_normal_fp16.safetensors",
   "https://huggingface.co/Comfy-Org/MoGe/resolve/main/geometry_estimation/moge_2_vitl_normal_fp16.safetensors", 0.6),

  ("acestep", "diffusion_models", "acestep_v1.5_turbo.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/diffusion_models/acestep_v1.5_turbo.safetensors", 4.5),
  ("acestep", "text_encoders", "qwen_0.6b_ace15.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_0.6b_ace15.safetensors", 1.1),
  ("acestep", "text_encoders", "qwen_1.7b_ace15.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/text_encoders/qwen_1.7b_ace15.safetensors", 3.5),
  ("acestep", "vae", "ace_1.5_vae.safetensors",
   "https://huggingface.co/Comfy-Org/ace_step_1.5_ComfyUI_files/resolve/main/split_files/vae/ace_1.5_vae.safetensors", 0.3),

  ("minimax-music", "diffusion_models", "minimax_music3_dit_fp16.safetensors",
   "https://huggingface.co/Comfy-Org/MiniMax-Music-3/resolve/main/diffusion_models/minimax_music3_dit_fp16.safetensors", 4.6),
  ("minimax-music", "vae", "minimax_music3_dav.safetensors",
   "https://huggingface.co/Comfy-Org/MiniMax-Music-3/resolve/main/vae/minimax_music3_dav.safetensors", 0.2),

  ("stable-audio", "checkpoints", "stable_audio_3_medium.safetensors",
   "https://huggingface.co/Comfy-Org/stable-audio-3/resolve/main/checkpoints/stable_audio_3_medium.safetensors", 8.6),
  ("stable-audio", "text_encoders", "t5gemma_b_b_ul2.safetensors",
   "https://huggingface.co/Comfy-Org/stable-audio-3/resolve/main/text_encoders/t5gemma_b_b_ul2.safetensors", 1.1),

  ("qwen3.5", "text_encoders", "qwen3.5_2b_bf16.safetensors",
   "https://huggingface.co/Comfy-Org/Qwen3.5/resolve/main/text_encoders/qwen3.5_2b_bf16.safetensors", 4.2),
]


def downloadModel(url, destdir, filename=None):
  dest = f"{WORKSPACE}/models/{destdir}"
  os.makedirs(dest, exist_ok=True)
  if filename is None:
    filename = url.split('/')[-1].removesuffix('?download=true')
  if 'civitai.com' in url and CIVITAI_API_TOKEN and 'token=' not in url:
    url = f"{url}?token={CIVITAI_API_TOKEN}"
  if os.path.exists(f"{dest}/{filename}"):
    print("skip (đã có):", filename)
    return
  !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest}" -o "{filename}"


selected = [m for m in MODELS if WANT.get(m[0], False)]
total = sum(m[4] for m in selected)
print(f"Sẽ tải {len(selected)} file (~{total:.1f} GB). Dùng DRIVE_PATH trên Drive để khỏi tải lại sau khi runtime reset.\n")

for group, destdir, filename, url, gb in selected:
  print(f"[{group}] {filename} (~{gb}G) -> models/{destdir}")
  downloadModel(url, destdir, filename)

# Tải thêm link bất kỳ (huggingface/civitai):
# downloadModel('https://civitai.com/api/download/models/2071650', 'checkpoints')  # cần CIVITAI_API_TOKEN
# downloadModel('https://huggingface.co/<repo>/resolve/main/model.safetensors', 'checkpoints')

# Custom nodes (uncomment để cài):
# !git clone https://github.com/ltdrdata/ComfyUI-Manager /content/ComfyUI/custom_nodes/ComfyUI-Manager
# !git clone https://github.com/ltdrdata/ComfyUI-Impact-Pack /content/ComfyUI/custom_nodes/ComfyUI-Impact-Pack


## LIST MODELS

In [ ]:
!find ./models -type f -not -name 'put_*' -exec du -h {} + | sort -rh | head -40


# `START ComfyUI  & Expose Server (MANUAL)`

## Install cloudflared

In [ ]:
# Cài cloudflared từ repo chính thức của Cloudflare (Debian/Ubuntu)
import os

SUDO = "" if os.geteuid() == 0 else "sudo"

!{SUDO} mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | {SUDO} tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | {SUDO} tee /etc/apt/sources.list.d/cloudflared.list
!{SUDO} apt-get update && {SUDO} apt-get install -y cloudflared
!cloudflared --version


In [ ]:
!python main.py --dont-print-server

# `START ComfyUI & Expose Server`

## Install cloudflared

In [ ]:
# Cài cloudflared từ repo chính thức của Cloudflare (Debian/Ubuntu)
import os

SUDO = "" if os.geteuid() == 0 else "sudo"

!{SUDO} mkdir -p --mode=0755 /usr/share/keyrings
!curl -fsSL https://pkg.cloudflare.com/cloudflare-main.gpg | {SUDO} tee /usr/share/keyrings/cloudflare-main.gpg >/dev/null
!echo 'deb [signed-by=/usr/share/keyrings/cloudflare-main.gpg] https://pkg.cloudflare.com/cloudflared any main' | {SUDO} tee /etc/apt/sources.list.d/cloudflared.list
!{SUDO} apt-get update && {SUDO} apt-get install -y cloudflared
!cloudflared --version


## CF Tunnel

In [ ]:
# Cloudflare named tunnel (không dùng quick tunnel / trycloudflare.com)
import subprocess
import threading
import time
import socket

from google.colab import userdata

CLOUDFLARE_TUNNEL_TOKEN = userdata.get('CLOUDFLARE_TUNNEL_TOKEN')
if not CLOUDFLARE_TUNNEL_TOKEN:
  raise ValueError(
      "Thiếu Colab secret 'CLOUDFLARE_TUNNEL_TOKEN'. Trong Cloudflare Zero Trust -> Networks -> "
      "Tunnels -> <tunnel của bạn> -> 'Install and run a connector' -> copy token."
  )

TUNNEL_HOSTNAME = ""  # @param {type:"string"}  # hostname public đã map trong Cloudflare, vd: comfy.example.com
PORT = 8188


def wait_port(port, timeout=600):
  deadline = time.time() + timeout
  while time.time() < deadline:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
      sock.settimeout(1)
      if sock.connect_ex(('127.0.0.1', port)) == 0:
        return True
    time.sleep(0.5)
  return False


def tunnel_thread(port):
  if not wait_port(port):
    print("\nComfyUI chưa sẵn sàng sau 10 phút - tunnel không được khởi động.\n")
    return
  print("\nComfyUI đã sẵn sàng, khởi động Cloudflare Tunnel (named tunnel)...\n")
  p = subprocess.Popen(
      ["cloudflared", "tunnel", "--no-autoupdate", "run", "--token", CLOUDFLARE_TUNNEL_TOKEN],
      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
  for line in p.stdout:
    l = line.decode(errors='replace')
    if "Registered tunnel connection" in l:
      print("Tunnel đã đăng ký. Mở ComfyUI tại: https://" +
            (TUNNEL_HOSTNAME or "<hostname đã cấu hình trong Cloudflare>"))
    print(l, end='')


threading.Thread(target=tunnel_thread, daemon=True, args=(PORT,)).start()

%cd {WORKSPACE}
!python main.py --dont-print-server
